# **Data Preprocessing for MIMIC-III**

Code adapted from https://github.com/LuChang-CS/CGL

In [2]:
import os
import pickle as pickle
import numpy as np
from datetime import datetime
import pandas as pd
import scipy.sparse as sps
import torch
from copy import deepcopy
import torch.nn as nn
import torch.nn.init as init
from torch.nn import functional as F
from collections import OrderedDict
import torch.utils.data as data
from torch.utils.data import DataLoader
from sklearn.model_selection import train_test_split
import random
import warnings
warnings.filterwarnings("ignore")

In [1]:
import os
import glob

base = '../data/MIMIC_III'
for ext in ('*.pkl', '*.npy'):
    for f in glob.glob(os.path.join(base, ext)):
        os.remove(f)
        print('deleted:', f)

In [3]:
def parse_admission(path) -> dict:
    print('parsing ADMISSIONS.csv ...')
    admission_path = os.path.join(path, 'ADMISSIONS.csv')

    admissions = pd.read_csv(admission_path)
    admissions.columns = admissions.columns.str.upper()

    admissions = admissions[['SUBJECT_ID', 'HADM_ID', 'ADMITTIME']].copy()
    admissions['SUBJECT_ID'] = admissions['SUBJECT_ID'].astype(int)
    admissions['HADM_ID'] = admissions['HADM_ID'].astype(int)
    admissions['ADMITTIME'] = admissions['ADMITTIME'].astype(str)

    all_patients = {}
    for _, row in admissions.iterrows():
        pid = row['SUBJECT_ID']
        admission_id = row['HADM_ID']
        admission_time = datetime.strptime(row['ADMITTIME'], '%Y-%m-%d %H:%M:%S')

        all_patients.setdefault(pid, []).append({
            'admission_id': admission_id,
            'admission_time': admission_time
        })

    patient_admission = {}
    for pid, admissions_ in all_patients.items():
        if len(admissions_) > 1:
            patient_admission[pid] = sorted(admissions_, key=lambda x: x['admission_time'])

    return patient_admission

In [4]:
def parse_diagnoses(path, patient_admission: dict) -> dict:
    print('parsing DIAGNOSES_ICD.csv ...')
    diagnoses_path = os.path.join(path, 'DIAGNOSES_ICD.csv')

    diagnoses = pd.read_csv(diagnoses_path)
    diagnoses.columns = diagnoses.columns.str.upper()

    diagnoses = diagnoses[['SUBJECT_ID', 'HADM_ID', 'ICD9_CODE']].copy()
    diagnoses['SUBJECT_ID'] = diagnoses['SUBJECT_ID'].astype(int)
    diagnoses['HADM_ID'] = diagnoses['HADM_ID'].astype(int)
    diagnoses['ICD9_CODE'] = diagnoses['ICD9_CODE'].astype(str).str.strip()

    valid_admissions = set()
    for pid, admissions in patient_admission.items():
        for adm in admissions:
            valid_admissions.add(adm['admission_id'])

    admission_codes = {}
    for _, row in diagnoses.iterrows():
        pid = row['SUBJECT_ID']
        hadm = row['HADM_ID']
        code = row['ICD9_CODE']

        if code in {'', 'nan', 'NaN', 'None'}:
            continue
        if pid not in patient_admission:
            continue
        if hadm not in valid_admissions:
            continue

        admission_codes.setdefault(hadm, []).append(code)

    return admission_codes

In [5]:
def calibrate_patient_by_admission(patient_admission: dict, admission_codes: dict):
    print('calibrating patients by admission ...')
    del_pids = []
    for pid, admissions in patient_admission.items():
        for admission in admissions:
            if admission['admission_id'] not in admission_codes:
                break
        else:
            continue
        del_pids.append(pid)
    for pid in del_pids:
        admissions = patient_admission[pid]
        for admission in admissions:
            if admission['admission_id'] in admission_codes:
                del admission_codes[admission['admission_id']]
        del patient_admission[pid]

In [6]:
raw_path = '../data/RAW/MIMIC_III/'
patient_admission = parse_admission(raw_path)
admission_codes = parse_diagnoses(raw_path, patient_admission)
calibrate_patient_by_admission(patient_admission, admission_codes)
print('There are %d valid patients' % len(patient_admission))

parsing ADMISSIONS.csv ...
parsing DIAGNOSES_ICD.csv ...
calibrating patients by admission ...
There are 14 valid patients


In [7]:
with open('../data/MIMIC_III/patient_admission.pkl', 'wb') as f155:
    pickle.dump(patient_admission, f155)

with open('../data/MIMIC_III/admission_codes.pkl', 'wb') as f156:
    pickle.dump(admission_codes, f156)

In [8]:
max_admission_num = 0
for pid, admissions in patient_admission.items():
    if len(admissions) > max_admission_num:
        max_admission_num = len(admissions)
max_code_num_in_a_visit = 0
for admission_id, codes in admission_codes.items():
    if len(codes) > max_code_num_in_a_visit:
        max_code_num_in_a_visit = len(codes)

In [9]:
def encode_code(admission_codes: dict) -> (dict, dict):
    print('encoding code ...')
    code_map = dict()
    for i, (admission_id, codes) in enumerate(admission_codes.items()):
        for code in codes:
            if code not in code_map:
                code_map[code] = len(code_map) + 1

    admission_codes_encoded = {
        admission_id: [code_map[code] for code in codes]
        for admission_id, codes in admission_codes.items()
    }
    return admission_codes_encoded, code_map

In [10]:
def encode_time_duration(patient_admission: dict) -> dict:
    print('encoding time duration ...')
    patient_time_duration_encoded = dict()
    for pid, admissions in patient_admission.items():
        duration = [0]
        for i in range(1, len(admissions)):
            days = (admissions[i]['admission_time'] - admissions[i - 1]['admission_time']).days
            duration.append(days)
        patient_time_duration_encoded[pid] = duration
    return patient_time_duration_encoded

In [11]:
def split_patients(patient_admission: dict, admission_codes: dict, code_map: dict, seed=6669):
    print('splitting train and test pids')

    pids = np.array(list(patient_admission.keys()))
    if len(pids) < 2:
        raise ValueError('Need at least 2 patients for a non-empty train/test split.')

    n_test = max(1, int(round(len(pids) * 0.2)))
    n_test = min(n_test, len(pids) - 1)

    train_pids, test_pids = train_test_split(
        pids,
        test_size=n_test,
        random_state=seed,
        shuffle=True
    )
    return np.array(train_pids), np.array(test_pids)

In [12]:
admission_codes_encoded, code_map = encode_code(admission_codes)
patient_time_duration_encoded = encode_time_duration(patient_admission)

code_num = len(code_map)

train_pids, test_pids = split_patients(
    patient_admission=patient_admission,
    admission_codes=admission_codes,
    code_map=code_map
)

encoding code ...
encoding time duration ...
splitting train and test pids


In [13]:
with open('../data/MIMIC_III/code_map.pkl', 'wb') as f13:
    pickle.dump(code_map, f13)

with open('../data/MIMIC_III/admission_codes_encoded.pkl', 'wb') as f157:
    pickle.dump(admission_codes_encoded, f157)

with open('../data/MIMIC_III/patient_time_duration_encoded.pkl', 'wb') as f158:
    pickle.dump(patient_time_duration_encoded, f158)

with open('../data/MIMIC_III/train_pids.npy', 'wb') as f258:
    np.save(f258, train_pids)

with open('../data/MIMIC_III/test_pids.npy', 'wb') as f259:
    np.save(f259, test_pids)

In [14]:
def build_code_xy(pids: np.ndarray,
                  patient_admission: dict,
                  admission_codes_encoded: dict,
                  max_admission_num: int,
                  code_num: int,
                  max_code_num_in_a_visit: int) -> (np.ndarray, np.ndarray, np.ndarray):
    print('building train/valid/test codes features and labels ...')
    n = len(pids)
    x = np.zeros((n, max_admission_num, max_code_num_in_a_visit), dtype=int)
    y = np.zeros((n, code_num), dtype=int)
    lens = np.zeros((n, ), dtype=int)
    for i, pid in enumerate(pids):
        print('\r\t%d / %d' % (i + 1, len(pids)), end='')
        admissions = patient_admission[pid]
        for k, admission in enumerate(admissions[:-1]):
            codes = admission_codes_encoded[admission['admission_id']]
            x[i][k][:len(codes)] = codes
        codes = np.array(admission_codes_encoded[admissions[-1]['admission_id']]) - 1
        y[i][codes] = 1
        lens[i] = len(admissions) - 1
    print('\r\t%d / %d' % (len(pids), len(pids)))
    return x, y, lens

In [15]:
def build_time_duration_xy(pids: np.ndarray,
                           patient_time_duration_encoded: dict,
                           max_admission_num: int) -> (np.ndarray, np.ndarray):
    print('building train/valid/test time duration features and labels ...')
    n = len(pids)
    x = np.zeros((n, max_admission_num))
    y = np.zeros((n, ))
    for i, pid in enumerate(pids):
        print('\r\t%d / %d' % (i + 1, len(pids)), end='')
        duration = patient_time_duration_encoded[pid]
        x[i][:len(duration) - 1] = duration[:-1]
        y[i] = duration[-1]
    print('\r\t%d / %d' % (len(pids), len(pids)))
    return x, y

In [16]:
train_codes_x, train_codes_y, train_visit_lens = build_code_xy(train_pids, patient_admission, admission_codes_encoded, max_admission_num, code_num, max_code_num_in_a_visit)
test_codes_x, test_codes_y, test_visit_lens = build_code_xy(test_pids, patient_admission, admission_codes_encoded, max_admission_num, code_num, max_code_num_in_a_visit)

building train/valid/test codes features and labels ...
	11 / 11
building train/valid/test codes features and labels ...
	3 / 3


In [17]:
with open('../data/MIMIC_III/train_codes_y.npy', 'wb') as f2:
    np.save(f2, train_codes_y)

with open('../data/MIMIC_III/train_visit_lens.npy', 'wb') as f3:
    np.save(f3, train_visit_lens)

with open('../data/MIMIC_III/test_codes_y.npy', 'wb') as f5:
    np.save(f5, test_codes_y)

with open('../data/MIMIC_III/test_visit_lens.npy', 'wb') as f6:
    np.save(f6, test_visit_lens)
    
with open('../data/MIMIC_III/train_codes_x.npy', 'wb') as f8:
    np.save(f8, train_codes_x)

with open('../data/MIMIC_III/test_codes_x.npy', 'wb') as f9:
    np.save(f9, test_codes_x)

In [18]:
def parse_icd9_range(range_: str) -> (str, str, int, int):
    ranges = range_.lstrip().split('-')
    if ranges[0][0] == 'V':
        prefix = 'V'
        format_ = '%02d'
        start, end = int(ranges[0][1:]), int(ranges[1][1:])
    elif ranges[0][0] == 'E':
        prefix = 'E'
        format_ = '%03d'
        start, end = int(ranges[0][1:]), int(ranges[1][1:])
    else:
        prefix = ''
        format_ = '%03d'
        if len(ranges) == 1:
            start = int(ranges[0])
            end = start + 1
        else:
            start, end = int(ranges[0]), int(ranges[1])
    return prefix, format_, start, end

In [19]:
def generate_code_levels(path, code_map: dict) -> np.ndarray:
    print('generating code levels ...')

    three_level_code_set = set(code.split('.')[0] for code in code_map)
    icd9_path = os.path.join(path, 'icd9.txt')
    icd9_range = list(open(icd9_path, 'r', encoding='utf-8').readlines())

    three_level_dict = dict()
    level1, level2, level3 = (1, 1, 1)
    level1_can_add = False

    for range_ in icd9_range:
        range_ = range_.rstrip()
        if not range_:
            continue

        if range_[0] == ' ':
            prefix, format_, start, end = parse_icd9_range(range_)
            level2_cannot_add = True

            for i in range(start, end + 1):
                code = prefix + format_ % i
                if code in three_level_code_set:
                    three_level_dict[code] = [level1, level2, level3]
                    level3 += 1
                    level1_can_add = True
                    level2_cannot_add = False

            if not level2_cannot_add:
                level2 += 1
        else:
            if level1_can_add:
                level1 += 1
                level1_can_add = False

    unknown_l1 = level1 + 1
    unknown_l2 = level2 + 1
    unknown_l3 = level3 + 1

    level4 = 1
    code_level = {}
    unknown_count = 0

    for code in code_map:
        three_level_code = code.split('.')[0]
        if three_level_code in three_level_dict:
            three_level = three_level_dict[three_level_code]
            code_level[code] = three_level + [level4]
        else:
            unknown_count += 1
            code_level[code] = [unknown_l1, unknown_l2, unknown_l3, level4]
        level4 += 1

    code_level_matrix = np.zeros((len(code_map) + 1, 4), dtype=int)
    for code, cid in code_map.items():
        code_level_matrix[cid] = code_level[code]

    print('unmapped three-level codes:', unknown_count)
    return code_level_matrix

In [20]:
def generate_patient_code_adjacent(code_x: np.ndarray, code_num: int) -> np.ndarray:
    print('generating patient code adjacent matrix ...')
    result = np.zeros((len(code_x), code_num + 1), dtype=int)
    for i, codes in enumerate(code_x):
        adj_codes = codes[codes > 0]
        result[i][adj_codes] = 1
    return result

In [21]:
def generate_code_code_adjacent(code_num: int, code_level_matrix: np.ndarray) -> np.ndarray:
    print('generating code code adjacent matrix ...')
    n = code_num + 1
    result = np.zeros((n, n), dtype=int)
    for i in range(1, n):
        print('\r\t%d / %d' % (i, n), end='')
        for j in range(1, n):
            if i != j:
                level_i = code_level_matrix[i]
                level_j = code_level_matrix[j]
                same_level = 4
                while same_level > 0:
                    level = same_level - 1
                    if level_i[level] == level_j[level]:
                        break
                    same_level -= 1
                result[i, j] = same_level + 1
    print('\r\t%d / %d' % (n, n))
    return result

In [22]:
def co_occur(pids: np.ndarray,
             patient_admission: dict,
             admission_codes_encoded: dict,
             code_num: int) -> (np.ndarray, np.ndarray, np.ndarray):
    print('calculating co-occurrence ...')
    x = np.zeros((code_num + 1, code_num + 1), dtype=float)
    for i, pid in enumerate(pids):
        print('\r\t%d / %d' % (i + 1, len(pids)), end='')
        admissions = patient_admission[pid]
        for k, admission in enumerate(admissions[:-1]):
            codes = admission_codes_encoded[admission['admission_id']]
            for m in range(len(codes) - 1):
                for n in range(m + 1, len(codes)):
                    c_i, c_j = codes[m], codes[n]
                    x[c_i, c_j] = 1
                    x[c_j, c_i] = 1
    print('\r\t%d / %d' % (len(pids), len(pids)))
    return x

In [23]:
l1 = len(train_pids)
train_patient_ids = np.arange(0, l1)
l2 = l1 + 0
l3 = l2 + len(test_pids)
test_patient_ids = np.arange(l2, l3)
pid_map = dict()
for i, pid in enumerate(train_pids):
    pid_map[pid] = train_patient_ids[i]
for i, pid in enumerate(test_pids):
    pid_map[pid] = test_patient_ids[i]

In [24]:
with open('../data/MIMIC_III/pid_map.pkl', 'wb') as f133:
    pickle.dump(pid_map, f133)

In [25]:
data_path = raw_path
code_levels = generate_code_levels(data_path, code_map)
patient_code_adj = generate_patient_code_adjacent(code_x=train_codes_x, code_num=code_num)
code_code_adj_t = generate_code_code_adjacent(code_level_matrix=code_levels, code_num=code_num)
co_occur_matrix = co_occur(train_pids, patient_admission, admission_codes_encoded, code_num)
code_code_adj = code_code_adj_t * co_occur_matrix

generating code levels ...
unmapped three-level codes: 251
generating patient code adjacent matrix ...
generating code code adjacent matrix ...
	260 / 260
calculating co-occurrence ...
	11 / 11


In [26]:
code_levels = code_levels[1:][:]                      # code_levels --> Remove first row
patient_code_adj = np.delete(patient_code_adj, 0, 1)  # patient_code_adj --> Remove first column
code_code_adj = np.delete(code_code_adj[1:][:], 0, 1) # code_code_adj --> Remove first row & column

In [27]:
binary_train_codes_x = []
for i in range(len(train_pids)):
    one_patient = np.zeros((train_visit_lens[i], code_num))
    for ii in range(train_visit_lens[i]):
        temp = train_codes_x[i][ii]
        temp = temp[temp > 0] - 1
        one_patient[ii][temp] = 1
    binary_train_codes_x.append(one_patient)

binary_test_codes_x = []
for i in range(len(test_pids)):
    one_patient = np.zeros((test_visit_lens[i], code_num))
    for ii in range(test_visit_lens[i]):
        temp = test_codes_x[i][ii]
        temp = temp[temp > 0] - 1
        one_patient[ii][temp] = 1
    binary_test_codes_x.append(one_patient)

In [28]:
with open('../data/MIMIC_III/patient_code_adj.npy', 'wb') as f11:
    np.save(f11, patient_code_adj)

with open('../data/MIMIC_III/code_levels.npy', 'wb') as f10:
    np.save(f10, code_levels)

with open('../data/MIMIC_III/code_code_adj.npy', 'wb') as f12:
    np.save(f12, code_code_adj)
    
with open('../data/MIMIC_III/binary_train_codes_x.pkl', 'wb') as f134:
    pickle.dump(binary_train_codes_x, f134)
    
with open('../data/MIMIC_III/binary_test_codes_x.pkl', 'wb') as f135:
    pickle.dump(binary_test_codes_x, f135)

In [29]:
import numpy as np
import pickle

base = '../data/MIMIC_III'

with open(f'{base}/binary_train_codes_x.pkl', 'rb') as f:
    train_x = pickle.load(f)
with open(f'{base}/binary_test_codes_x.pkl', 'rb') as f:
    test_x = pickle.load(f)

train_pids = np.load(f'{base}/train_pids.npy')
test_pids = np.load(f'{base}/test_pids.npy')
code_levels = np.load(f'{base}/code_levels.npy')

print('train_x =', len(train_x))
print('test_x =', len(test_x))
print('train_pids =', len(train_pids))
print('test_pids =', len(test_pids))
print('rows with non-positive code levels =', ((code_levels <= 0).any(axis=1)).sum())
print('min code level per column =', code_levels.min(axis=0))

train_x = 11
test_x = 3
train_pids = 11
test_pids = 3
rows with non-positive code levels = 0
min code level per column = [1 1 1 1]
